In [ ]:
import os
from tqdm import tqdm
import re
import sys
from flask import Flask, request, jsonify, send_file
from werkzeug.utils import secure_filename
import zipfile
import io

app = Flask(__name__)

# 配置上传文件的目录
UPLOAD_FOLDER = 'uploads'
SPLIT_FOLDER = 'split_chapters'
ALLOWED_EXTENSIONS = {'txt'}

# 创建必要的目录
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(SPLIT_FOLDER, exist_ok=True)

app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 限制上传文件大小为16MB

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

def get_novel_name(file_path):
    """
    从文件路径中提取小说名称
    
    参数:
        file_path (str): 小说文件路径
    
    返回:
        str: 小说名称
    """
    # 获取文件名（不含扩展名）
    file_name = os.path.basename(file_path)
    novel_name = os.path.splitext(file_name)[0]
    return novel_name

def is_chapter_title(text):
    """
    判断文本是否为章节标题
    
    参数:
        text (str): 待判断的文本
    
    返回:
        tuple: (is_title, title_type)
            - is_title: 是否为章节标题
            - title_type: 标题类型
    """
    # 章节标题检测
    patterns = [
        # 卷章节格式
        r'^第[一二三四五六七八九十百千万\d]+卷\s*《[^》]+》',
        r'^第\d+卷\s*《[^》]+》',
        r'^第[一二三四五六七八九十百千万\d]+卷\s*[^（(]+',
        r'^第\d+卷\s*[^（(]+',
        
        # 卷章连体格式
        r'^第[一二三四五六七八九十百千万\d]+卷第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+卷第\d+章\s*[^（(]+',
        r'^第[一二三四五六七八九十百千万\d]+卷\s*第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+卷\s*第\d+章\s*[^（(]+',
        
        # 基本章节格式
        r'^第[一二三四五六七八九十百千万\d]+章\s*[^（(]+（[一二三四五六七八九十\d]+）',
        r'^第[一二三四五六七八九十百千万\d]+章\s*[^(]+\([一二三四五六七八九十\d]+\)',
        r'^第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+章\s*[^（(]+',
        
        # 回目格式
        r'^第[一二三四五六七八九十百千万\d]+回\s*[^（(]+',
        r'^第\d+回\s*[^（(]+',
        
        # 特殊章节
        r'^楔子\s*[^（(]*',
        r'^序章\s*[^（(]*',
        r'^尾声\s*[^（(]*',
        
        # 英文格式
        r'^Chapter\s*\d+\s*[^(]*',
        
        # 数字格式
        r'^[一二三四五六七八九十\d]+[、.]\s*[^（(]*',
        r'^\s*[一二三四五六七八九十\d]+[、.]\s*[^（(]*',
        
        # 带引号的格式
        r'^『[一二三四五六七八九十\d]+』\s*[^（(]*',
        r'^「[一二三四五六七八九十\d]+」\s*[^（(]*',
        r'^《[一二三四五六七八九十\d]+》\s*[^（(]*',
        
        # 带括号的格式
        r'^（[一二三四五六七八九十\d]+）\s*[^（(]*',
        r'^\([一二三四五六七八九十\d]+\)\s*[^（(]*',
        
        # 其他常见格式
        r'^\s*第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^\s*第\d+章\s*[^（(]+',
        r'^\s*第[一二三四五六七八九十百千万\d]+回\s*[^（(]+',
        r'^\s*第\d+回\s*[^（(]+',
    ]
    
    # 检查是否是卷标题
    volume_patterns = [
        r'^第[一二三四五六七八九十百千万\d]+卷\s*《[^》]+》',
        r'^第\d+卷\s*《[^》]+》',
        r'^第[一二三四五六七八九十百千万\d]+卷\s*[^（(]+',
        r'^第\d+卷\s*[^（(]+',
        r'^\s*第[一二三四五六七八九十百千万\d]+卷\s*[^（(]+',
        r'^\s*第\d+卷\s*[^（(]+',
    ]
    
    # 检查是否是卷章连体格式
    volume_chapter_patterns = [
        r'^第[一二三四五六七八九十百千万\d]+卷第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+卷第\d+章\s*[^（(]+',
        r'^第[一二三四五六七八九十百千万\d]+卷\s*第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+卷\s*第\d+章\s*[^（(]+',
        r'^\s*第[一二三四五六七八九十百千万\d]+卷\s*第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^\s*第\d+卷\s*第\d+章\s*[^（(]+',
    ]
    
    # 首先检查是否是卷章连体格式
    for pattern in volume_chapter_patterns:
        if re.match(pattern, text):
            return True, 'volume_chapter'
    
    # 然后检查是否是卷标题
    for pattern in volume_patterns:
        if re.match(pattern, text):
            return True, 'volume'
    
    # 最后检查其他格式
    for pattern in patterns:
        if re.match(pattern, text):
            return True, 'chapter'
    
    return False, 'other'

def split_novel(novel_path, base_output_dir='split_chapters'):
    """
    将小说分割成单独的章节文件
    
    参数:
        novel_path (str): 小说文件路径
        base_output_dir (str): 基础输出目录
    
    返回:
        tuple: (success, message, chapter_count, output_dir)
    """
    try:
        # 检查输入文件是否存在
        if not os.path.exists(novel_path):
            return False, f"错误：小说文件不存在 - {novel_path}", 0, None
            
        # 获取小说名称
        novel_name = get_novel_name(novel_path)
        
        # 创建小说专属的输出目录
        output_dir = os.path.join(base_output_dir, novel_name)
        os.makedirs(output_dir, exist_ok=True)
        
        # 读取小说文件
        with open(novel_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        # 处理每一行
        current_volume = None
        current_chapter = None
        current_content = []
        chapter_count = 0
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            # 检查是否是章节标题
            is_title, title_type = is_chapter_title(line)
            
            if is_title:
                # 保存之前的章节内容
                if current_chapter is not None and current_content:
                    # 创建章节文件
                    if title_type == 'volume_chapter':
                        chapter_file = os.path.join(output_dir, f"{current_chapter}.txt")
                    elif current_volume:
                        chapter_file = os.path.join(output_dir, f"{current_volume}_{current_chapter}.txt")
                    else:
                        chapter_file = os.path.join(output_dir, f"{current_chapter}.txt")
                    try:
                        with open(chapter_file, 'w', encoding='utf-8') as f:
                            f.write('\n'.join(current_content))
                        chapter_count += 1
                    except Exception as e:
                        return False, f"保存章节时出错: {e}", chapter_count, output_dir
                
                # 开始新的章节
                if title_type == 'volume':
                    current_volume = line
                    current_chapter = None
                    current_content = []
                elif title_type == 'volume_chapter':
                    current_chapter = line
                    current_content = []
                else:
                    current_chapter = line
                    current_content = []
            else:
                # 添加内容到当前章节
                current_content.append(line)
        
        # 保存最后一个章节
        if current_chapter is not None and current_content:
            if title_type == 'volume_chapter':
                chapter_file = os.path.join(output_dir, f"{current_chapter}.txt")
            elif current_volume:
                chapter_file = os.path.join(output_dir, f"{current_volume}_{current_chapter}.txt")
            else:
                chapter_file = os.path.join(output_dir, f"{current_chapter}.txt")
            try:
                with open(chapter_file, 'w', encoding='utf-8') as f:
                    f.write('\n'.join(current_content))
                chapter_count += 1
            except Exception as e:
                return False, f"保存最后一个章节时出错: {e}", chapter_count, output_dir
        
        return True, f"成功处理完成，共分割{chapter_count}个章节", chapter_count, output_dir
        
    except Exception as e:
        return False, f"处理过程中出错: {e}", 0, None

@app.route('/split', methods=['POST'])
def split_novel_api():
    """
    处理小说分章API端点
    """
    # 检查是否有文件被上传
    if 'file' not in request.files:
        return jsonify({'success': False, 'message': '没有上传文件'}), 400
    
    file = request.files['file']
    if file.filename == '':
        return jsonify({'success': False, 'message': '没有选择文件'}), 400
    
    if not allowed_file(file.filename):
        return jsonify({'success': False, 'message': '不支持的文件类型'}), 400
    
    try:
        # 保存上传的文件
        filename = secure_filename(file.filename)
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(filepath)
        
        # 处理小说分章
        success, message, chapter_count, output_dir = split_novel(filepath)
        
        if not success:
            return jsonify({
                'success': False,
                'message': message
            }), 500
        
        # 创建ZIP文件
        memory_file = io.BytesIO()
        with zipfile.ZipFile(memory_file, 'w', zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files in os.walk(output_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_name = os.path.relpath(file_path, output_dir)
                    zf.write(file_path, arc_name)
        
        memory_file.seek(0)
        
        # 清理临时文件
        os.remove(filepath)
        
        return send_file(
            memory_file,
            mimetype='application/zip',
            as_attachment=True,
            download_name=f'{os.path.splitext(filename)[0]}_chapters.zip'
        )
        
    except Exception as e:
        return jsonify({
            'success': False,
            'message': f'处理过程中出错: {str(e)}'
        }), 500

@app.route('/', methods=['GET'])
def index():
    """
    返回简单的API使用说明
    """
    return '''
    <html>
        <head>
            <title>小说分章服务</title>
            <style>
                body { font-family: Arial, sans-serif; margin: 40px; }
                .container { max-width: 800px; margin: 0 auto; }
                h1 { color: #333; }
                form { margin: 20px 0; }
                .note { color: #666; }
            </style>
        </head>
        <body>
            <div class="container">
                <h1>小说分章服务</h1>
                <form action="/split" method="post" enctype="multipart/form-data">
                    <input type="file" name="file" accept=".txt">
                    <input type="submit" value="上传并分章">
                </form>
                <div class="note">
                    <p>说明：</p>
                    <ul>
                        <li>请上传TXT格式的小说文件</li>
                        <li>文件大小限制为16MB</li>
                        <li>处理完成后将自动下载分章结果的ZIP压缩包</li>
                    </ul>
                </div>
            </div>
        </body>
    </html>
    '''

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000) 

小说名称: 超级兵王
输出目录: split_chapters\超级兵王
正在读取小说内容...
共读取到 75084 行内容

前10行内容:
行 1: 《超级兵王》全集
行 2: 
行 3: 作者：明朝无酒
行 4: 
行 5: 
行 6: 简介
行 7: 
行 8: 一个从中国最著名的秦城监狱里走出来的男人，于嬉笑怒骂中一路高歌猛进，挟腥风血雨横扫一切。他的格言是——这个世界必将因为我而颤抖！
行 9: 
行 10: 第一卷第一章狱锁狂龙


处理小说内容:   3%|▎         | 2320/75084 [00:00<00:03, 22723.64it/s]


发现标题: 第一卷第一章狱锁狂龙 (类型: volume_chapter)

发现标题: 第一卷第二章姐姐花钱不心疼 (类型: volume_chapter)
已保存章节: 第一卷第一章狱锁狂龙

发现标题: 第一卷第三章家 (类型: volume_chapter)
已保存章节: 第一卷第二章姐姐花钱不心疼

发现标题: 第一卷第四章请我喝一杯 (类型: volume_chapter)
已保存章节: 第一卷第三章家

发现标题: 第一卷第五章米兰姐 (类型: volume_chapter)
已保存章节: 第一卷第四章请我喝一杯

发现标题: 第一卷第六章林公子 (类型: volume_chapter)
已保存章节: 第一卷第五章米兰姐

发现标题: 第一卷第七章同事 (类型: volume_chapter)
已保存章节: 第一卷第六章林公子

发现标题: 第一卷第八章新员工 (类型: volume_chapter)
已保存章节: 第一卷第七章同事

发现标题: 第一卷第九章公车色狼 (类型: volume_chapter)
已保存章节: 第一卷第八章新员工

发现标题: 第一卷第十章你三围多少 (类型: volume_chapter)
已保存章节: 第一卷第九章公车色狼

发现标题: 第一卷第十一章任务 (类型: volume_chapter)
已保存章节: 第一卷第十章你三围多少

发现标题: 第一卷第十二章庭院深深 (类型: volume_chapter)
已保存章节: 第一卷第十一章任务

发现标题: 第一卷第十三章帅哥，玩一下吧 (类型: volume_chapter)
已保存章节: 第一卷第十二章庭院深深

发现标题: 第一卷第十四章找不着家的美女 (类型: volume_chapter)
已保存章节: 第一卷第十三章帅哥，玩一下吧

发现标题: 第一卷第十五章去我家吧 (类型: volume_chapter)
已保存章节: 第一卷第十四章找不着家的美女

发现标题: 第一卷第十六章好白好大 (类型: volume_chapter)
已保存章节: 第一卷第十五章去我家吧

发现标题: 第一卷第十七章小两口 (类型: volume_chapter)
已保存章节: 第一卷第十六章好白好大

发现标题: 第一卷第十八章我们是一家人 (类型: volume_ch

处理小说内容:   9%|▉         | 7017/75084 [00:00<00:03, 22668.92it/s]

已保存章节: 第一卷第三十七章美酒佳人

发现标题: 第一卷第三十九章有钱人 (类型: volume_chapter)
已保存章节: 第一卷第三十八章业务不熟练

发现标题: 第一卷第四十章叶蓓蓓的爱心晚餐 (类型: volume_chapter)
已保存章节: 第一卷第三十九章有钱人

发现标题: 第一卷第四十一章警察上门 (类型: volume_chapter)
已保存章节: 第一卷第四十章叶蓓蓓的爱心晚餐

发现标题: 第一卷第四十二章好福气 (类型: volume_chapter)
已保存章节: 第一卷第四十一章警察上门

发现标题: 第一卷第四十三章将逛街进行到底 (类型: volume_chapter)
已保存章节: 第一卷第四十二章好福气

发现标题: 第一卷第四十四章专家 (类型: volume_chapter)
已保存章节: 第一卷第四十三章将逛街进行到底

发现标题: 第一卷第四十五章你适合丁字裤 (类型: volume_chapter)
已保存章节: 第一卷第四十四章专家

发现标题: 第一卷第四十六章计划 (类型: volume_chapter)
已保存章节: 第一卷第四十五章你适合丁字裤

发现标题: 第一卷第四十七章夜话 (类型: volume_chapter)
已保存章节: 第一卷第四十六章计划

发现标题: 第一卷第四十八章姑爷上门 (类型: volume_chapter)
已保存章节: 第一卷第四十七章夜话

发现标题: 第一卷第四十九章几年不见 (类型: volume_chapter)
已保存章节: 第一卷第四十八章姑爷上门

发现标题: 第一卷第五十章宋家明 (类型: volume_chapter)
已保存章节: 第一卷第四十九章几年不见

发现标题: 第一卷第五十一章投缘 (类型: volume_chapter)
已保存章节: 第一卷第五十章宋家明

发现标题: 第一卷第五十二章国之九鼎 (类型: volume_chapter)
已保存章节: 第一卷第五十一章投缘

发现标题: 第一卷第五十三章姐妹花回上海了 (类型: volume_chapter)
已保存章节: 第一卷第五十二章国之九鼎

发现标题: 第一卷第五十四章都是贵宾 (类型: volume_chapter)
已保存章节: 第一卷第五十三章姐妹花回上海了

发现标题

...

已保存章节: 第一卷第四百九十一章反宴

发现标题: 第一卷第四百九十三章你说该不该揍 (类型: volume_chapter)
已保存章节: 第一卷第四百九十二章拥抱一下

发现标题: 第一卷第四百九十四章何必再自责 (类型: volume_chapter)
已保存章节: 第一卷第四百九十三章你说该不该揍

发现标题: 第一卷第四百九十五章拍戏遇险 (类型: volume_chapter)
已保存章节: 第一卷第四百九十四章何必再自责

发现标题: 第一卷第四百九十六章加薪，压压惊 (类型: volume_chapter)
已保存章节: 第一卷第四百九十五章拍戏遇险

发现标题: 第一卷第四百九十七章齐聚日本 (类型: volume_chapter)
已保存章节: 第一卷第四百九十六章加薪，压压惊

发现标题: 第一卷第四百九十八章各自谋划 (类型: volume_chapter)
已保存章节: 第一卷第四百九十七章齐聚日本

发现标题: 第一卷第四百九十九章军舰漏水了 (类型: volume_chapter)
已保存章节: 第一卷第四百九十八章各自谋划

发现标题: 第一卷第五百章震动 (类型: volume_chapter)
已保存章节: 第一卷第四百九十九章军舰漏水了

发现标题: 第一卷第五百零一章动手 (类型: volume)
已保存章节: 第一卷第五百章震动

发现标题: 第一卷第五百零二章全面进攻 (类型: volume)

发现标题: 第一卷第五百零三章玉碎 (类型: volume)

发现标题: 第一卷第五百零四章我要你当天皇 (类型: volume)

发现标题: 第一卷第五百零五章神秘箱子 (类型: volume)

发现标题: 第一卷第五百零六章如封似闭 (类型: volume)

发现标题: 第一卷第五百零七章白袍老人 (类型: volume)

发现标题: 第一卷第五百零八章联手 (类型: volume)

发现标题: 第一卷第五百零九章声明 (类型: volume)

发现标题: 第一卷第五百一十章掌控皇室 (类型: volume_chapter)

发现标题: 第一卷第五百一十一章嘴仗 (类型: volume_chapter)
已保存章节: 第一卷第五百一十章掌控皇室

发现标题: 第一卷第五百一十二章悔不当年 (类型: volum

处理小说内容:  97%|█████████▋| 72673/75084 [00:02<00:00, 23370.31it/s]

已保存章节: 第一卷第五百三十六章美国人的内裤

发现标题: 第一卷第五百三十八章吓死人的徽章 (类型: volume_chapter)
已保存章节: 第一卷第五百三十七章喀秋莎号游轮

发现标题: 第一卷第五百三十九章宫廷宴会 (类型: volume_chapter)
已保存章节: 第一卷第五百三十八章吓死人的徽章

发现标题: 第一卷第五百四十章兄弟决斗 (类型: volume_chapter)
已保存章节: 第一卷第五百三十九章宫廷宴会

发现标题: 第一卷第五百四十一章司徒 (类型: volume_chapter)
已保存章节: 第一卷第五百四十章兄弟决斗

发现标题: 第一卷第五百四十二章美国大乱 (类型: volume_chapter)
已保存章节: 第一卷第五百四十一章司徒

发现标题: 第一卷第五百四十三章极乐城 (类型: volume_chapter)
已保存章节: 第一卷第五百四十二章美国大乱

发现标题: 第一卷第五百四十四章钻石 (类型: volume_chapter)
已保存章节: 第一卷第五百四十三章极乐城

发现标题: 第一卷第五百四十五章赌场 (类型: volume_chapter)
已保存章节: 第一卷第五百四十四章钻石

发现标题: 第一卷第五百四十六章这里的钞票是废纸 (类型: volume_chapter)
已保存章节: 第一卷第五百四十五章赌场

发现标题: 第一卷第五百四十七章恩怨 (类型: volume_chapter)
已保存章节: 第一卷第五百四十六章这里的钞票是废纸

发现标题: 第一卷第五百四十八章普京设宴 (类型: volume_chapter)
已保存章节: 第一卷第五百四十七章恩怨

发现标题: 第一卷第五百四十九章共同的话题 (类型: volume_chapter)
已保存章节: 第一卷第五百四十八章普京设宴

发现标题: 第一卷第五百五十章希望我做些什么 (类型: volume_chapter)
已保存章节: 第一卷第五百四十九章共同的话题

发现标题: 第一卷第五百五十一章遇记者 (类型: volume_chapter)
已保存章节: 第一卷第五百五十章希望我做些什么

发现标题: 第一卷第五百五十二章采访 (类型: volume_chapter)
已保存章节: 第一卷第五百五十一章遇记者

发现标题

处理小说内容: 100%|██████████| 75084/75084 [00:02<00:00, 25409.90it/s]

已保存章节: 第一卷第五百六十九章选择

发现标题: 第一卷第五百七十一章谁的圈套 (类型: volume_chapter)
已保存章节: 第一卷第五百七十章你爸是不是李刚

发现标题: 第一卷第五百七十二章省长算个…… (类型: volume_chapter)
已保存章节: 第一卷第五百七十一章谁的圈套

发现标题: 第一卷第五百七十三章洪天马 (类型: volume_chapter)
已保存章节: 第一卷第五百七十二章省长算个……

发现标题: 第一卷第五百七十四章我要你做我的棋子 (类型: volume_chapter)
已保存章节: 第一卷第五百七十三章洪天马

发现标题: 第一卷第五百七十五章当狗的资格都没 (类型: volume_chapter)
已保存章节: 第一卷第五百七十四章我要你做我的棋子

发现标题: 第一卷第五百七十六章唯他独尊 (类型: volume_chapter)
已保存章节: 第一卷第五百七十五章当狗的资格都没
已保存章节: 第一卷第五百七十六章唯他独尊

处理完成！共保存了 528 个章节文件
小说分割完成，章节文件保存在: split_chapters\超级兵王
请检查输出目录中的文件。
